In [10]:
import json
import random
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from tqdm import tqdm


# request로 부터 crawl하는 코드

In [11]:
BASE = "https://www.daangn.com"

UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0.0.0 Safari/537.36"
)

HEADERS: dict[str, str] = {
    "User-Agent": UA,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
    "Referer": BASE,
}


REMIX_RE = re.compile(
    r"window\.__remixContext\s*=\s*(\{.*?\})\s*;\s*</script>",
    re.S
)


def extract_remix_context(html: str) -> dict:
    m = REMIX_RE.search(html)
    if not m:
        raise ValueError("remixContext not found")

    return json.loads(m.group(1))


def dig(obj: object, path: list[str]) -> object | None:

    cur = obj

    for p in path:

        if not isinstance(cur, dict) or p not in cur:
            return None

        cur = cur[p]

    return cur


def parse_detail(html: str) -> dict[str, object | None]:

    remix = extract_remix_context(html)

    loader = dig(remix, ["state", "loaderData"])

    if not isinstance(loader, dict):
        raise ValueError("loaderData missing")

    route_key = next(
        (k for k in loader if "buy_sell_id" in k),
        None
    )

    if route_key is None:
        raise ValueError("route key missing")

    product = dig(loader, [route_key, "product"])

    if not isinstance(product, dict):
        raise ValueError("product missing")

    user = product.get("user", {})

    images = product.get("images")

    image_count = len(images) if isinstance(images, list) else None

    return {

        "favoriteCount": product.get("favoriteCount"),
        "chatCount": product.get("chatCount"),
        "viewCount": product.get("viewCount"),
        "sellerTemperature": user.get("score"),
        "imageCount": image_count,

    }


def fetch_detail(
    session: requests.Session,
    href: str,
    cookie: str | None = None,
) -> dict[str, object | None]:

    url = urljoin(BASE, href)

    headers = HEADERS.copy()

    if cookie:
        headers["Cookie"] = cookie

    r = session.get(url, headers=headers, timeout=20)

    r.raise_for_status()

    return parse_detail(r.text)


def enrich_df_with_detail(
    df: pd.DataFrame,
    href_col: str = "href",
    cookie: str | None = None,
    sleep_min: float = 1.0,
    sleep_max: float = 2.5,
    retries: int = 3
) -> pd.DataFrame:

    rows: list[dict] = []

    urls = df[href_col].astype(str).tolist()

    with requests.Session() as session:

        for href in tqdm(urls, desc="Crawling detail pages"):

            result = {
                "favoriteCount": None,
                "chatCount": None,
                "viewCount": None,
                "sellerTemperature": None,
                "imageCount": None,
            }

            last_error = None

            for attempt in range(retries):

                try:

                    data = fetch_detail(session, href, cookie)

                    result.update(data)

                    break

                except Exception as e:

                    last_error = e

                    time.sleep(1.5 * (attempt + 1))

            rows.append(result)

            time.sleep(random.uniform(sleep_min, sleep_max))

    detail_df = pd.DataFrame(rows)

    merged = pd.concat([df.reset_index(drop=True), detail_df], axis=1)

    return merged

In [ ]:
# 쿠키가 있어야 서버로부터 block당하지 않을 확률이 증가함
cookie = r'_gcl_au=1.1.1444086516.1772091281; _ga=GA1.1.1864578218.1772091281; _fbp=fb.1.1772091280970.177480178187518519; _clck=1v9lldb%5E2%5Eg42%5E0%5E2248; search_region=eyJyZWdpb25JZCI6IjYwMzQiLCJyZWdpb25OYW1lIjoi7IK87ISx64%2BZIiwiY291bnRyeUNvZGUiOiJrciJ9; ph_phc_FqtkYFQ4JIDKz1sAraoJA02LALWTkJh8F8okhjnxd3Z_posthog=%7B%22distinct_id%22%3A%22019c98df-0c6b-78ed-9837-b90a124b3fa1%22%2C%22%24sesid%22%3A%5B1772602141904%2C%22019cb71e-cfb7-7a46-8fa5-c5e46c3f17f5%22%2C1772598775734%5D%2C%22%24initial_person_info%22%3A%7B%22r%22%3A%22https%3A%2F%2Fwww.google.com%2F%22%2C%22u%22%3A%22https%3A%2F%2Fwww.daangn.com%2Fkr%2F%22%7D%7D; _clsk=1ot3q4y%5E1772602142308%5E34%5E0%5Es.clarity.ms%2Fcollect; _ga_KNCHQ0QW6Q=GS2.1.s1772598776$o10$g1$t1772602142$j49$l0$h0'

In [7]:
def bundle_json_to_df(path: str) -> pd.DataFrame:
    with open(path, encoding='utf-8') as f:
        bundle = json.load(f)

    rows = []
    for a in bundle.get('articles', []):
        user = a.get('user') or {}
        region = a.get('region') or {}
        meta = a.get('_meta') or {}

        rows.append(
            {
                'id': a.get('id').split('-')[-1][:-1],
                'href': a.get('href'),
                'price': a.get('price'),
                'title': a.get('title'),
                'status': a.get('status'),
                'content': a.get('content'),
                'createdAt': a.get('createdAt'),
                'boostedAt': a.get('boostedAt'),
                'user_dbId': user.get('dbId'),
                'user_nickname': user.get('nickname'),
                'region_name_from_article': region.get('name'),
                # ✅ 관리용(요청 지역 정보)
                'region_id': meta.get('region_id'),
                'region_name': meta.get('region_name'),
                'region_in': meta.get('region_in'),
                'crawledAt': meta.get('crawledAt'),
            }
        )

    df = pd.DataFrame(rows)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df['createdAt'] = pd.to_datetime(df['createdAt'], errors='coerce')
    df['boostedAt'] = pd.to_datetime(df['boostedAt'], errors='coerce')
    df['crawledAt'] = pd.to_datetime(df['crawledAt'], errors='coerce').dt.tz_localize(
        'Asia/Seoul'
    )
    for col in ['createdAt', 'boostedAt', 'crawledAt']:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df[col] = df[col].dt.tz_convert('Asia/Seoul')
    return df

In [ ]:
keyword = '폴로'

# json파일의 데이터를 df로 불러오기
df = bundle_json_to_df(f'../data/json/search_{keyword}.json')
df.head(1)

,id,href,price,title,status,content,createdAt,boostedAt,user_dbId,user_nickname,region_name_from_article,region_id,region_name,region_in,crawledAt
0,8wqyok7b9sf6,https://www.daangn.com/kr/buy-sell/%ED%8F%B4%E...,71000.0,폴로 랄프로렌 에스테트립 하프 집업팝니다.,Ongoing,구입 후 보관만 한 새상품급 에스테이트립 집업입니다.\n백화점 구입 정품입니다.\n...,2026-02-25 20:51:02.197000+09:00,2026-03-03 17:12:18.814000+09:00,11016644,당근당근당근,삼성동,6034,삼성동,삼성동-6034,2026-03-03 17:19:55+09:00


In [ ]:
# 파싱한 데이터를 새로운 데이터프레임에 합치기
df2 = enrich_df_with_detail(
    df, 
    href_col='href', 
    cookie=cookie,
    sleep_min=1.5,
    sleep_max=3.5
)

# 데이터 parquet으로 저장 (json보다 20배 이상 빠르고 10배 작음)
df2.to_parquet(f"../data/parquet/daangn_{keyword}.parquet")

Crawling detail pages:   3%|▎         | 273/8970 [13:56<6:05:15,  2.52s/it] 